# LangChain Hands-on 11 — RAG — Chat with your PDF
**Trainer:** Ajeetkumar

**What we will practise**
- RAG steps end-to-end
- Load → Split → Embed → Store → Retrieve → Generate
- RAG chain with LCEL
- Answer with sources
- Say “I don’t know” when the answer is not in the document
- Conversational RAG (with history)

> Run on Google Colab. Add your Groq key in Colab **Secrets** (🔑 icon) with the name `groq_key` (free key: console.groq.com).

In [ ]:
! pip install -q langchain langchain-groq langchain-community langchain-huggingface sentence-transformers faiss-cpu pypdf

In [ ]:
from google.colab import userdata
groq_api_key = userdata.get('groq_key')

In [ ]:
import os
os.environ['GROQ_API_KEY'] = groq_api_key

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-20b")

# What is RAG?
**Retrieval-Augmented Generation** = give the LLM the right pages of your document at question time.

```
INDEXING (once)
PDF ─▶ Load ─▶ Split ─▶ Embed ─▶ Vector Store

QUESTION TIME (every query)
Question ─▶ Retriever ─▶ top chunks ─▶ Prompt(context + question) ─▶ LLM ─▶ Answer + sources
```
**Why?** LLM doesn't know your private data · reduces hallucination · no fine-tuning needed · data can be updated anytime.

## Step 1 — Load

In [ ]:
! wget -q -O attention.pdf https://arxiv.org/pdf/1706.03762

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
pages = PyPDFLoader("attention.pdf").load()
len(pages)

## Step 2 — Split

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150).split_documents(pages)
len(chunks)

## Step 3 — Embed + Step 4 — Store

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)

## Step 5 — Retrieve

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

In [ ]:
for d in retriever.invoke("What is multi-head attention?"):
    print("page", d.metadata["page"], "→", d.page_content[:120], "\n")

## Step 6 — Generate (the RAG chain)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer ONLY from the context below. "
               "If the answer is not in the context, say: I don't know based on the document.\n\n"
               "Context:\n{context}"),
    ("human", "{question}"),
])

In [ ]:
def format_docs(docs):
    return "\n\n".join(f"[page {d.metadata['page']}] {d.page_content}" for d in docs)

In [ ]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
print(rag_chain.invoke("What is multi-head attention?"))

In [ ]:
print(rag_chain.invoke("What BLEU score did the Transformer get on English-to-German?"))

In [ ]:
print(rag_chain.invoke("Who won the cricket world cup 2011?"))       # not in the PDF → should say I don't know

## Answer + sources (for citations in UI)

In [ ]:
from langchain_core.runnables import RunnableParallel

In [ ]:
rag_with_sources = RunnableParallel(
    {"docs": retriever, "question": RunnablePassthrough()}
).assign(answer=(lambda x: {"context": format_docs(x["docs"]), "question": x["question"]}) | prompt | llm | StrOutputParser())

In [ ]:
result = rag_with_sources.invoke("Why do they use positional encoding?")
print(result["answer"])
print("\nSources: pages", sorted({d.metadata["page"] for d in result["docs"]}))

## Streaming RAG answer

In [ ]:
for chunk in rag_chain.stream("Explain scaled dot-product attention simply"):
    print(chunk, end="", flush=True)

## Conversational RAG — follow-up questions
Problem: "what about its training time?" — *its* = ? → we rewrite the question using history first.

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain.messages import HumanMessage, AIMessage

In [ ]:
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", "Rewrite the last user question as a standalone question using the chat history. Only return the question."),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])
rewrite_chain = rewrite_prompt | llm | StrOutputParser()

In [ ]:
history = []

def ask(question):
    standalone = rewrite_chain.invoke({"history": history, "question": question}) if history else question
    answer = rag_chain.invoke(standalone)
    history.extend([HumanMessage(question), AIMessage(answer)])
    print("🔎 searched:", standalone)
    return answer

In [ ]:
print(ask("What model architecture does the paper propose?"))

In [ ]:
print(ask("How long did it take to train it?"))

# RAG checklist
- ✅ Good chunk size (500–1000) + overlap
- ✅ Same embedding model for docs & questions
- ✅ "Answer only from context" + "I don't know" rule
- ✅ Return sources (page numbers)
- ✅ Hybrid search / re-ranking for better retrieval (notebook 10)
- ✅ Evaluate: is the answer **faithful** to the context? is the context **relevant**?

## 🧪 Practice
1. Replace the PDF with your company policy / resume and ask 5 questions.
2. Use the hybrid retriever (BM25 + FAISS) inside the RAG chain.
3. Build a Gradio UI: `gr.ChatInterface(lambda msg, hist: ask(msg)).launch()`.